In [1]:
%%capture --no-stderr
! pip install -U langgraph firecrawl-py -q

In [2]:
import getpass
from typing import List, Optional, TypedDict, Any
from typing_extensions import Annotated
from operator import or_
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from IPython.display import Image, display
import logging

In [6]:
# === Configuração de registro ===
logger = logging.getLogger()
logger.setLevel(logging.INFO) # Defina o nível do registrador raiz como INFO

# Remova todos os manipuladores associados ao objeto logger raiz.
if logger.hasHandlers():
    logger.handlers.clear()

# Crie um novo manipulador
handler = logging.StreamHandler()
handler.setLevel(logging.INFO)  # Defina o nível do manipulador como INFO

# Crie um formatador e defina-o para o manipulador
formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
handler.setFormatter(formatter)

# Adicione o manipulador ao registrador raiz
logger.addHandler(handler)
# === Fim da configuração de registro ===

In [7]:
FIRECRAWL_API_KEY = getpass.getpass("Firecrawl API Key: ")

Firecrawl API Key: ··········


In [8]:
def first_non_null(a, b):
    return a if a is not None else b

In [9]:
# Defina o esquema InputState
class InputState(TypedDict):
    url: str
    keyword: str

# Defina o esquema OverallState com redutores
class OverallState(TypedDict):
    urls: List[str]
    current_url_index: int
    total_urls: int
    urls_to_scrape: List[str]
    extracted_info: Annotated[Optional[str], first_non_null]
    extracted_from_url: Annotated[Optional[str], first_non_null]
    is_information_found: Annotated[Optional[bool], or_]
    keyword: str

In [10]:
def first_node(state: InputState) -> OverallState:
    logging.info("Executando nó: first_node")
    return {
        "urls": [state["url"]],
        "current_url_index": 0,
        "total_urls": 0,
        "urls_to_scrape": [],
        "keyword": state["keyword"],
        "extracted_info": None,
        "extracted_from_url": None,
        "is_information_found": False,
    }

In [11]:
from firecrawl import FirecrawlApp

def get_sitemap(state: OverallState, config):
    logging.info("Nó em execução: get_sitemap")
    api_key = config.get("configurable", {}).get("firecrawl_api_key")
    if not api_key:
        logging.error("A chave da API do Firecrawl está faltando na configuração.")
        raise ValueError("A chave da API do Firecrawl está faltando na configuração.")

    app = FirecrawlApp(api_key=api_key)

    # Mapeie o site para obter a lista de URLs
    map_result = app.map_url(state["urls"][0])
    logging.info(f"Resultado do mapa: {map_result}")

    # Manipule o novo formato de resposta que é um dicionário
    if isinstance(map_result, dict):
        if map_result.get('success') and isinstance(map_result.get('links'), list):
            sitemap = map_result['links']
            state["urls"] = sitemap
            state["total_urls"] = len(sitemap)
            logging.info(f"Encontrados {len(sitemap)} URLs para raspar.")
        else:
            logging.error(f"Error: Estrutura de resposta inválida em map_result: {map_result}")
            state["urls"] = []

    # Mantenha o tratamento da lista original para compatibilidade com versões anteriores
    elif isinstance(map_result, list) and len(map_result) > 0:
        sitemap = map_result
        state["urls"] = sitemap
        state["total_urls"] = len(sitemap)
        logging.info(f"Encontrados {len(sitemap)} URLs para raspar.")
    else:
        logging.error(f"Error: Formato inesperado em map_result: {map_result}")
        state["urls"] = []

    return state

/usr/local/lib/python3.10/dist-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "schema" in "FirecrawlApp.ExtractParams" shadows an attribute in parent "BaseModel"
  warnings.warn(


In [13]:
def scrape_manager(state: OverallState) -> OverallState:
    logging.info("Executando nó: scrape_manager")
    total_urls = state.get("total_urls", 0)
    current_index = state["current_url_index"]
    next_index = min(current_index + 3, total_urls)
    urls_to_scrape = state["urls"][current_index:next_index]
    state["current_url_index"] = next_index  # Atualizar o índice atual
    state["urls_to_scrape"] = urls_to_scrape  # Armazene os URLs para raspar

    if total_urls > 0:
        progress = (current_index / total_urls) * 100
        logging.info(f"Progresso: {current_index}/{total_urls} URLs processadas ({progress:.2f}%)")
    return state

# Função para retornar objetos de envio para nós scraper
def continue_to_scraper(state: OverallState):
    logging.info(f"Preparando para raspar URLs {state['current_url_index'] - len(state['urls_to_scrape'])} para {state['current_url_index'] - 1}")
    # Retornar lista de objetos Enviar para o nó 'raspador'
    return [Send("scraper", {"url": url, "keyword": state["keyword"]}) for url in state["urls_to_scrape"]]

In [14]:
def scraper(state: OverallState, config):
    url = state["url"]
    keyword = state["keyword"]
    logging.info(f"Executando nó: scraper para URL: {url}")
    api_key = config.get("configurable", {}).get("firecrawl_api_key")
    if not api_key:
        logging.error("A chave da API do Firecrawl está faltando na configuração.")
        raise ValueError("A chave da API do Firecrawl está faltando na configuração.")

    app = FirecrawlApp(api_key=api_key)

    # Raspe a URL e obtenha markdown
    try:
        logging.info(f"Começou a raspar: {url}")
        scrape_result = app.scrape_url(url, params={'formats': ['markdown']})
        logging.info(f"URL raspada: {url}")
    except Exception as e:
        logging.error(f"Ocorreu uma exceção ao extrair URL: {url}. Error: {e}")
        scrape_result = {}

    if isinstance(scrape_result, dict):
        markdown = scrape_result.get("markdown", "")
        # Verifique se a palavra-chave está em markdown
        if markdown and keyword in markdown:
            # Informação encontrada
            return {
                "extracted_info": markdown,
                "extracted_from_url": url,
                "is_information_found": True
            }
        else:
            # Informação não encontrada
            return {"is_information_found": False}
    else:
        logging.warning(f"Falha ao extrair URL: {url}")
        return {"is_information_found": False}

In [23]:
def evaluate(state: OverallState):
    logging.info("Executando nó: evaluate")
    # A lógica é tratada na função de roteamento
    return state

# Função de roteamento condicional para nó de avaliação
def evaluate_routing(state: OverallState):
    if state.get("is_information_found", False):
        logging.info(f"Informação encontrada na URL: {state.get('extracted_from_url')}")
        logging.info("Finalizando a execução do gráfico.")
        return "END"
    elif state["current_url_index"] < len(state["urls"]):
        logging.info("Informação não encontrada, continuando a raspar o próximo lote.")
        return "scrape_manager"
    else:
        logging.info("Não há mais URLs para raspar, Finalizando a execução do gráfico.")
        return "END"

In [24]:
# Construir o gráfico com esquema de configuração
class ConfigSchema(TypedDict):
    firecrawl_api_key: str

builder = StateGraph(OverallState, input=InputState, config_schema=ConfigSchema)

builder.add_node("first_node", first_node)
builder.add_node("get_sitemap", get_sitemap)
builder.add_node("scrape_manager", scrape_manager)
builder.add_node("scraper", scraper)
builder.add_node("evaluate", evaluate)

# Adicionar arestas entre nós
builder.add_edge(START, "first_node")
builder.add_edge("first_node", "get_sitemap")
builder.add_edge("get_sitemap", "scrape_manager")

# Adicionar arestas condicionais do scrape_manager aos nós do scraper
builder.add_conditional_edges("scrape_manager", continue_to_scraper, ["scraper"])

builder.add_edge("scraper", "evaluate")

# Adicionar aresta condicional do nó de avaliação com mapeamento explícito
builder.add_conditional_edges(
    "evaluate",
    evaluate_routing,
    {"END": END, "scrape_manager": "scrape_manager"}
)

# Compilar o gráfico
graph = builder.compile()

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
config = {"configurable": {"firecrawl_api_key": FIRECRAWL_API_KEY},
          "recursion_limit": 100}

input_state = {
    "url": "https://www.eb.mil.br/",
    "keyword": "Operação Perseu"
}

result = graph.invoke(input_state, config=config)

In [ ]:
result